In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
df=spark.read.format('csv')\
            .option("inferSchema", True)\
            .option('header', True)\
            .load(f"file:///home/talentum/test-jupyter/DAWPPS/json-usecases/labs/data/modified-movies.csv")
df.show()
df.printSchema()

+---+--------------------+----+-----+-----------+--------------------+
| id|               Title|Year|Rated|   Released|               Genre|
+---+--------------------+----+-----+-----------+--------------------+
|  1|              Avatar|2009|PG-13|18 Dec 2009|["Action","Advent...|
|  2|         I Am Legend|2007|PG-13|14 Dec 2007|["Drama","Horror"...|
|  3|                 300|2006|    R|09 Mar 2007|["Action","Drama"...|
|  4|        The Avengers|2012|PG-13|04 May 2012|["Action","Sci-Fi...|
|  5|The Wolf of Wall ...|2013|    R|25 Dec 2013|["Biography","Com...|
|  6|        Interstellar|2014|PG-13|07 Nov 2014|["Adventure","Dra...|
+---+--------------------+----+-----+-----------+--------------------+

root
 |-- id: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Rated: string (nullable = true)
 |-- Released: string (nullable = true)
 |-- Genre: string (nullable = true)



In [6]:
import pyspark.sql.functions as F

# Using the select 
# and to_json function to make create the json text field
# from different column 
# in to new data frame
df2=df.select(F.col("id"), F.to_json(\
 F.struct("Title",\
         "Year", \
         "Rated", \
         "Released",\
         "Genre"))\
        .alias("movie"))
df2.printSchema()
df2.show(truncate = False)

root
 |-- id: integer (nullable = true)
 |-- movie: string (nullable = true)

+---+-----------------------------------------------------------------------------------------------------------------------------------+
|id |movie                                                                                                                              |
+---+-----------------------------------------------------------------------------------------------------------------------------------+
|1  |{"Title":"Avatar","Year":2009,"Rated":"PG-13","Released":"18 Dec 2009","Genre":"[\"Action\",\"Adventure\",\"Fantasy\"]"}           |
|2  |{"Title":"I Am Legend","Year":2007,"Rated":"PG-13","Released":"14 Dec 2007","Genre":"[\"Drama\",\"Horror\",\"Sci-Fi\"]"}           |
|3  |{"Title":"300","Year":2006,"Rated":"R","Released":"09 Mar 2007","Genre":"[\"Action\",\"Drama\",\"Fantasy\"]"}                      |
|4  |{"Title":"The Avengers","Year":2012,"Rated":"PG-13","Released":"04 May 2012","Genre":"[\"